# Real-World Data Engineering ETL Pipeline (OOP & LLD)

This notebook implements a generic, Object-Oriented ETL pipeline based on real-world Data Engineering practices (Low-Level Design).

**Key Concepts Implemented:**
1. **Abstract Interfaces:** `Extractor`, `Transformer`, and `Loader` base classes to allow easily swapping components (e.g., swapping SQLite for PostgreSQL).
2. **Data Validation:** Pre-transformation schema and data quality checks.
3. **Robustness:** Error handling, timeouts, and detailed logging.
4. **Orchestration:** An `ETLPipeline` class to manage the flow.

In [ ]:
import requests
import pandas as pd
import sqlite3
import logging
from abc import ABC, abstractmethod
from typing import Any, Dict, List

# Setup Logging for observability
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s'
)

In [ ]:
class Extractor(ABC):
    @abstractmethod
    def extract(self, source: str) -> Any:
        """Extract data from a source system."""
        pass

class Transformer(ABC):
    @abstractmethod
    def validate(self, data: pd.DataFrame) -> pd.DataFrame:
        """Validate data schema and quality before transformation."""
        pass

    @abstractmethod
    def transform(self, data: Any) -> pd.DataFrame:
        """Apply business logic and transformations."""
        pass

class Loader(ABC):
    @abstractmethod
    def load(self, data: pd.DataFrame, target: str) -> None:
        """Load data into the target destination."""
        pass

In [ ]:
class RESTAPIExtractor(Extractor):
    def __init__(self, timeout: int = 10):
        self.timeout = timeout
        self.logger = logging.getLogger("RESTAPIExtractor")

    def extract(self, source: str) -> List[Dict]:
        self.logger.info(f"Starting extraction from API: {source}")
        try:
            response = requests.get(source, timeout=self.timeout)
            response.raise_for_status()
            data = response.json()
            self.logger.info(f"Successfully extracted {len(data)} records.")
            return data
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Failed to extract data from {source}: {e}")
            raise

In [ ]:
class BaseTransformer(Transformer):
    """Base class providing generic validation utilities."""
    def __init__(self):
        self.logger = logging.getLogger(self.__class__.__name__)

    def check_required_columns(self, df: pd.DataFrame, required_cols: List[str]):
        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            raise ValueError(f"Validation failed. Missing columns: {missing}")

class ProductTransformer(BaseTransformer):
    def validate(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.info("Validating Product Data...")
        self.check_required_columns(df, ['id', 'title', 'price', 'category'])

        # Data Quality: Drop missing critical fields and ensure price >= 0
        df = df.dropna(subset=['id', 'title', 'price'])
        df = df[df['price'] >= 0]
        return df

    def transform(self, data: List[Dict]) -> pd.DataFrame:
        df = pd.DataFrame(data)
        df = self.validate(df)
        self.logger.info("Transforming Product Data...")

        # Flatten nested rating dictionary
        if 'rating' in df.columns:
            df['rating_rate'] = df['rating'].apply(lambda x: x.get('rate') if isinstance(x, dict) else None)
            df['rating_count'] = df['rating'].apply(lambda x: x.get('count') if isinstance(x, dict) else None)
            df = df.drop(columns=['rating'])

        return df

class UserTransformer(BaseTransformer):
    def validate(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.info("Validating User Data...")
        self.check_required_columns(df, ['id', 'email', 'username'])

        # Data Quality: Regex Email Validation
        valid_email_mask = df['email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', na=False)
        df = df[valid_email_mask]
        return df

    def transform(self, data: List[Dict]) -> pd.DataFrame:
        df = pd.DataFrame(data)
        df = self.validate(df)
        self.logger.info("Transforming User Data...")

        # Flatten nested 'name' and 'address'
        if 'name' in df.columns:
            df['first_name'] = df['name'].apply(lambda x: x.get('firstname', '') if isinstance(x, dict) else '')
            df['last_name'] = df['name'].apply(lambda x: x.get('lastname', '') if isinstance(x, dict) else '')
            df = df.drop(columns=['name'])

        if 'address' in df.columns:
            df['city'] = df['address'].apply(lambda x: x.get('city', '') if isinstance(x, dict) else '')
            df['zipcode'] = df['address'].apply(lambda x: x.get('zipcode', '') if isinstance(x, dict) else '')
            df = df.drop(columns=['address'])

        return df

class CartTransformer(BaseTransformer):
    def validate(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.info("Validating Cart Data...")
        self.check_required_columns(df, ['id', 'userId', 'date', 'products'])
        return df

    def transform(self, data: List[Dict]) -> pd.DataFrame:
        df = pd.DataFrame(data)
        df = self.validate(df)
        self.logger.info("Transforming Cart Data...")

        # 1st Normal Form: Explode the products array
        df = df.explode('products').reset_index(drop=True)

        # Extract individual product details out of the exploded dictionaries
        df['product_id'] = df['products'].apply(lambda x: x.get('productId') if isinstance(x, dict) else None)
        df['quantity'] = df['products'].apply(lambda x: x.get('quantity') if isinstance(x, dict) else None)
        df = df.drop(columns=['products'])

        # Standardize Date format
        df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d %H:%M:%S')

        return df

In [ ]:
class SQLiteLoader(Loader):
    def __init__(self, db_path: str):
        self.db_path = db_path
        self.logger = logging.getLogger("SQLiteLoader")

    def load(self, df: pd.DataFrame, target_table: str) -> None:
        self.logger.info(f"Loading {len(df)} records into SQLite table: {target_table}")
        try:
            with sqlite3.connect(self.db_path) as conn:
                df.to_sql(target_table, conn, if_exists='replace', index=False)
            self.logger.info(f"Successfully loaded data into {target_table}.")
        except Exception as e:
            self.logger.error(f"Failed to load data into {target_table}: {e}")
            raise

In [ ]:
class ETLPipeline:
    """Orchestrates the execution of Extractor, Transformer, and Loader."""
    def __init__(self, name: str, extractor: Extractor, transformer: Transformer, loader: Loader):
        self.name = name
        self.extractor = extractor
        self.transformer = transformer
        self.loader = loader
        self.logger = logging.getLogger(f"Pipeline_{self.name}")

    def run(self, source_url: str, target_table: str):
        self.logger.info(f"--- Starting Pipeline: {self.name} ---")
        try:
            raw_data = self.extractor.extract(source_url)
            transformed_data = self.transformer.transform(raw_data)
            self.loader.load(transformed_data, target_table)
            self.logger.info(f"--- Pipeline {self.name} completed successfully! ---\n")
        except Exception as e:
            self.logger.error(f"--- Pipeline {self.name} failed: {e} ---\n")

In [ ]:
# Endpoints and Targets
PRODUCTS_API = "https://fakestoreapi.com/products/"
USERS_API = "https://fakestoreapi.com/users/"
CARTS_API = "https://fakestoreapi.com/carts/"
DATABASE_FILE = "fakestore_warehouse.db"

# Initialize Shared Dependencies (Dependency Injection)
api_extractor = RESTAPIExtractor()
db_loader = SQLiteLoader(DATABASE_FILE)

# 1. Run Product Pipeline
ETLPipeline(
    name="Products_ETL", extractor=api_extractor, transformer=ProductTransformer(), loader=db_loader
).run(PRODUCTS_API, "dim_products")

# 2. Run User Pipeline
ETLPipeline(
    name="Users_ETL", extractor=api_extractor, transformer=UserTransformer(), loader=db_loader
).run(USERS_API, "dim_users")

# 3. Run Cart Pipeline
ETLPipeline(
    name="Carts_ETL", extractor=api_extractor, transformer=CartTransformer(), loader=db_loader
).run(CARTS_API, "fact_carts")

print("All pipelines executed. Connecting to database to verify...")

# Verify Output
with sqlite3.connect(DATABASE_FILE) as conn:
    print("\n--- Fact Carts Sample ---")
    print(pd.read_sql("SELECT * FROM fact_carts LIMIT 3", conn))

All pipelines executed. Connecting to database to verify...

--- Fact Carts Sample ---
   id  userId                 date  __v  product_id  quantity
0   1       1  2020-03-02 00:00:00    0           1         4
1   1       1  2020-03-02 00:00:00    0           2         1
2   1       1  2020-03-02 00:00:00    0           3         6
